---
title: "Лабораторна робота №1. Попередня обробка текстових даних та формування корпусу"
description: "Технології аналізу текстової інформації та машинне навчання | КрНУ ім. М. Остроградського"
author: "&copy; Viacheslav Romenskyi, 2026"
date: today  # Автоматично вставить поточну дату у форматі YYYY-MM-DD
lang: uk
jupyter: python3

format:
  html:
    toc: true
    toc-location: right
    number-sections: false
    code-fold: true
    embed-resources: true
    self-contained-math: true
---

::: callout-note
## Примітка
Для виконання лабораторної роботи потрібно опрацювати матеріал лекції 3 і мати інстальований дистрибутив Miniconda. Див. налаштування у відповідній лабораторній роботі "Лабораторна робота №0. Налаштування Python/DS середовищ (Miniconda + Positron)"
:::

## Мета роботи

Опанувати базовий NLP-pipeline для українських текстів в середовищі Python: збирання та організацію корпусу, очищення тексту, токенізацію, видалення стоп-слів, просту нормалізацію та побудову базової статистики корпусу.

## 1, 2. Формування корпусу

Спочатку було додано корпуси, надані викладачем у лабораторній роботі і до них додані 20 текстів індивідуальної частини корпусу. Текти були обрані із різних статей щодо text mining та deep learning.

In [69]:
from pathlib import Path

base_corpus = [
    "Сучасні системи обробки природної мови використовують статистичні та нейромережеві методи.",
    "Машинне навчання дозволяє аналізувати великі корпуси текстових даних.",
    "Попередня обробка тексту включає очищення, токенізацію та нормалізацію.",
    "Українські тексти потребують коректної роботи зі стоп-словами та морфологією.",
    "Векторизація тексту є основою для класифікації, кластеризації та аналізу тональності."
]

def load_txt_corpus(folder_path):
    texts = []
    for file in Path(folder_path).glob("*.txt"):
        with open(file, encoding="utf-8") as f:
            texts.append(f.read())
    return texts

BASE_DIR = Path().resolve() # Get the current notebook path
txt_corpus = load_txt_corpus(BASE_DIR / "lab1" / "text")

corpus = base_corpus + txt_corpus
len(corpus)

25

## 3. Очищення тексту

Очищення передбачає:

- переведення тексту в нижній регістр;
- видалення URL;
- видалення зайвих символів і пунктуації;
- усунення зайвих пробілів.

In [70]:
import re # For regex

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^а-щьюяґєіїa-z0-9\s'-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned_corpus = [clean_text(text) for text in corpus]
cleaned_corpus[5]

'інтелектуальний аналіз тексту іат англ text mining напрям інтелектуального аналізу даних англ data mining та штучного інтелекту метою якого є отримання інформації з колекцій текстових документів ґрунтуючись на застосуванні ефективних у практичному плані методів машинного навчання та обробки природної мови інтелектуальний аналіз тексту використовує всі ті ж підходи до перероблювання інформації що й інтелектуальний аналіз даних однак різниця між цими напрямками проявляється лише в кінцевих методах а також у тому що інтелектуальний аналіз даних має справу зі сховищами та базами даних а не електронними бібліотеками та корпусами текстів'

## 4. Токенізація

Токенізація -- це розбиття тексту на окремі одиниці, зазвичай слова.

In [71]:
def token(text: str) -> list:
    return text.split()

tokenized_corpus = [token(text) for text in cleaned_corpus]
tokenized_corpus[5]

['інтелектуальний',
 'аналіз',
 'тексту',
 'іат',
 'англ',
 'text',
 'mining',
 'напрям',
 'інтелектуального',
 'аналізу',
 'даних',
 'англ',
 'data',
 'mining',
 'та',
 'штучного',
 'інтелекту',
 'метою',
 'якого',
 'є',
 'отримання',
 'інформації',
 'з',
 'колекцій',
 'текстових',
 'документів',
 'ґрунтуючись',
 'на',
 'застосуванні',
 'ефективних',
 'у',
 'практичному',
 'плані',
 'методів',
 'машинного',
 'навчання',
 'та',
 'обробки',
 'природної',
 'мови',
 'інтелектуальний',
 'аналіз',
 'тексту',
 'використовує',
 'всі',
 'ті',
 'ж',
 'підходи',
 'до',
 'перероблювання',
 'інформації',
 'що',
 'й',
 'інтелектуальний',
 'аналіз',
 'даних',
 'однак',
 'різниця',
 'між',
 'цими',
 'напрямками',
 'проявляється',
 'лише',
 'в',
 'кінцевих',
 'методах',
 'а',
 'також',
 'у',
 'тому',
 'що',
 'інтелектуальний',
 'аналіз',
 'даних',
 'має',
 'справу',
 'зі',
 'сховищами',
 'та',
 'базами',
 'даних',
 'а',
 'не',
 'електронними',
 'бібліотеками',
 'та',
 'корпусами',
 'текстів']

## 5. Видалення стоп-слів

Для лабораторної роботи використаємо розширений список стоп-слів.

In [72]:
uk_stopwords = {
    "і", "й", "та", "або", "чи",
    "в", "у", "на", "до", "з", "із", "зі", "за", "під", "над", "між",
    "про", "для", "при", "без", "через", "а",

    "я", "ти", "він", "вона", "воно", "ми", "ви", "вони",
    "мене", "тобі", "його", "її", "нас", "вас", "їх",
    "них", "тебе",

    "це", "той", "та", "те", "ці", "цей", "такий", "таке", "такі",

    "є", "був", "була", "було", "були", "буде", "будуть",
    "бути", "є", "нема", "немає",

    "як", "що", "щоб", "тому", "тому що", "бо",
    "коли", "де", "куди", "звідки",

    "дуже", "ще", "вже", "тільки", "лише", "навіть", "саме",
    "так", "ні", "ось", "от", "ну",

    "свій", "своя", "своє", "свої",
    "мій", "моя", "моє", "мої",
    "твій", "твоя", "твоє", "твої",

    "може", "можуть",

    "ж", "би", "б", "же", "не", "хоча",
    "який", "якого", "якої", "яких", "таких", "такого", "яка", "яке", "які"
}

def remove_stopwords(tokens: list) -> list:
    return [token for token in tokens if token not in uk_stopwords]

normalized_corpus = [remove_stopwords(tokens) for tokens in tokenized_corpus]
normalized_corpus[6]

['більшість',
 'сучасних',
 'моделей',
 'глибокого',
 'навчання',
 'ґрунтуються',
 'багатошарових',
 'штучних',
 'нейронних',
 'мережах',
 'згорткові',
 'нейронні',
 'мережі',
 'трансформери',
 'також',
 'належати',
 'пропозиційні',
 'формули',
 'латентні',
 'змінні',
 'організовані',
 'пошарово',
 'глибоких',
 'породжувальних',
 'моделях',
 'вузли',
 'глибоких',
 'мережах',
 'переконань',
 'глибоких',
 'машинах',
 'больцмана']

## 6. Формування корпусу для аналізу

Після очищення, токенізації та видалення стоп-слів отримуємо корпус у формі списку документів, де кожен документ подано списком токенів.

In [73]:
final_corpus = normalized_corpus

## 7. Базова статистика корпусу

На цьому етапі обчислюємо:

- кількість документів;
- середню довжину документа;
- розмір словника корпусу;
- найчастотніші слова.

In [74]:
from collections import Counter

doc_number = len(final_corpus)
doc_lengths = [len(text) for text in final_corpus]
all_tokens = [token for doc in final_corpus for token in doc]
distinct_tokens = sorted(set(all_tokens))
freq = Counter(all_tokens)

print("Кількість документів:", doc_number)
print("Загальний розмір документів:", sum(doc_lengths))
print("Середній розмір документа:", sum(doc_lengths) / doc_number)
print("Розмір словнику:", len(distinct_tokens))
print("20 найчастотніших слів:")
print(freq.most_common(20))

Кількість документів: 25
Загальний розмір документів: 871
Середній розмір документа: 34.84
Розмір словнику: 560
20 найчастотніших слів:
[('навчання', 27), ('даних', 19), ('аналіз', 11), ('глибоке', 10), ('тексту', 8), ('штучного', 7), ('інтелекту', 7), ('інформації', 7), ('мережі', 7), ('текстових', 6), ('мережах', 6), ('мови', 5), ('дозволяє', 5), ('інтелектуальний', 5), ('документів', 5), ('машинного', 5), ('глибокого', 5), ('великих', 5), ('документи', 5), ('машинне', 4)]


## 8. Порівняння тексту до і після обробки

Цей крок важливий для інтерпретації результату preprocessing.

In [75]:
i = 5
print("Оригінал:")
print(corpus[i])
print("\nПісля очищення:")
print(cleaned_corpus[i])
print("\nПісля нормалізації:")
print(final_corpus[i])

Оригінал:
Інтелектуальний аналіз тексту (ІАТ, англ. text mining) — напрям інтелектуального аналізу даних (англ. Data Mining) та штучного інтелекту, метою якого є отримання інформації з колекцій текстових документів, ґрунтуючись на застосуванні ефективних, у практичному плані, методів машинного навчання та обробки природної мови. Інтелектуальний аналіз тексту використовує всі ті ж підходи до перероблювання інформації, що й інтелектуальний аналіз даних, однак різниця між цими напрямками проявляється лише в кінцевих методах, а також у тому, що інтелектуальний аналіз даних має справу зі сховищами та базами даних, а не електронними бібліотеками та корпусами текстів.

Після очищення:
інтелектуальний аналіз тексту іат англ text mining напрям інтелектуального аналізу даних англ data mining та штучного інтелекту метою якого є отримання інформації з колекцій текстових документів ґрунтуючись на застосуванні ефективних у практичному плані методів машинного навчання та обробки природної мови інтеле

## Висновки

Під час виконання лабораторної роботи було сформовано корпус із 20 текстів на тему text mining та deep learning. Спочатку тексти пройшли очищення, що дозволило прибрали усі посилання, всі символи, крім обраних, зайві пробіли та інші whitespaces. Після цього тексти були поділені на токени - слова, для подального аналізу. Із тексту далі були прибрані усі стоп-слова, які заважають дослітити корпус через відсутність корисної інформації, що вони несуть. Список стоп-слів був розширений для того, аби краще підходити під задачу. Отже, після preprocessing тексти набули вигляду, що позбавляє нас непотрібної для аналізу інформації. Для нашого корпусу було виявлено кількість документів - 25, загальну кількість слів - 871, розмір словника - 560, середній розмір документа - 34.84. Найпопулярнішими словами є 'навчання' (27), 'даних' (19), 'аналіз' (11), 'глибоке' (10) та 'тексту' (8).

## Контрольні питання

1. Що таке корпус текстових даних?
Корпус текстових даних — це структурована колекція текстів, зібрана для аналізу або навчання моделей. Він може містити документи різного типу та обсягу, об’єднані спільною тематикою або задачею.
2. Яка роль попередньої обробки тексту в NLP?
Попередня обробка очищає та нормалізує текст, роблячи його придатним для аналізу моделями. Вона підвищує якість результатів і зменшує шум у даних.
3. Що таке токенізація?
Токенізація — це процес розбиття тексту на менші одиниці (слова чи речення). Вона є першим кроком у більшості задач обробки тексту.
4. Для чого видаляють стоп-слова?
Стоп-слова видаляють, щоб прибрати часті, але малозмістовні слова, які не впливають на сенс тексту. Це допомагає зменшити розмір даних і покращити ефективність моделей.
5. Які базові характеристики корпусу можна обчислити після preprocessing?
Можна визначити кількість документів, слів і унікальних токенів. Також обчислюють частоти слів, середню довжину текстів і розмір словника.